In [1]:
import numpy as np
import pandas as pd

TRAIN = 'data/train.csv'
TEST = 'data/test.csv'

train = pd.read_csv(TRAIN)
test = pd.read_csv(TEST)

print(f'Training data shape: {train.shape}')
print(f'Missing values in training data: {train.isnull().sum()}')
print(f'Test data shape: {test.shape}')
print(train.head())

Training data shape: (159571, 8)
Missing values in training data: id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64
Test data shape: (153164, 2)
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0      

In [2]:
cols_drop = ['id']
train.drop(columns=cols_drop, inplace=True)
test_ids = test['id']
test.drop(columns=cols_drop, inplace=True)

import re

def clean_text(text):
    # text = text.lower()
    # text = re.sub(r'[^a-z0-9\s ]', '', text)
    # text = re.sub(r'\s+', ' ', text).strip()
    return text

train['comment_text'] = train['comment_text'].apply(clean_text)
test['comment_text'] = test['comment_text'].apply(clean_text)

print(train['comment_text'][0])

Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_val = train_test_split(train, test_size=0.2, random_state=42)
y_train = X_train.drop(columns=['comment_text'])
y_val = X_val.drop(columns=['comment_text'])

X_train = X_train['comment_text']
X_val = X_val['comment_text']

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train)
X_val = vectorizer.transform(X_val)

print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5570585 stored elements and shape (127656, 165785)>
  Coords	Values
  (0, 64604)	0.6354351137360599
  (0, 144103)	0.5903394834391763
  (0, 132284)	0.0641219733550259
  (0, 27803)	0.15791403543877752
  (0, 73680)	0.03936618086035522
  (0, 148068)	0.32063614689207104
  (0, 76943)	0.036198030973445115
  (0, 67632)	0.12080238515765755
  (0, 68668)	0.09815467610009292
  (0, 146631)	0.030835935680660877
  (0, 68399)	0.11940003147633074
  (0, 6857)	0.13384969350017153
  (0, 6954)	0.1451122224880219
  (0, 7032)	0.13433054315334111
  (0, 5193)	0.12865743895130177
  (1, 73680)	0.040983900933826756
  (1, 76943)	0.037685558593500826
  (1, 146631)	0.1605155625359101
  (1, 92499)	0.07673361271721484
  (1, 2773)	0.1242186964947812
  (1, 153168)	0.08934771112788821
  (1, 77311)	0.0388694985845842
  (1, 160574)	0.19061910155529146
  (1, 21407)	0.13844836950036324
  (1, 49232)	0.18272255189045356
  :	:
  (127654, 111834)	0.20029422938843067
 

In [5]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import roc_auc_score

In [6]:
model_rf = OneVsRestClassifier(RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
))

model_rf.fit(X_train, y_train)

val_preds_rf = model_rf.predict_proba(X_val)

rf_auc = roc_auc_score(y_val, val_preds_rf)
print(f'Random Forest Validation AUC: {rf_auc}')

Random Forest Validation AUC: 0.9548896884842777


In [7]:
model_lr = OneVsRestClassifier(LogisticRegression(
    random_state=42, 
    solver='liblinear',
    C=3,
    max_iter=2000,
    n_jobs=-1,
    class_weight='balanced'
))

model_lr.fit(X_train, y_train)

val_preds_lr = model_lr.predict_proba(X_val)

auc_lr = roc_auc_score(y_val, val_preds_lr)
print(f'Validation ROC AUC (Logistic Regression): {auc_lr}')

d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\

Validation ROC AUC (Logistic Regression): 0.9794806546773641


In [8]:
model_xgb = OneVsRestClassifier(XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.3,
    n_jobs=-1,
))

model_xgb.fit(X_train, y_train)

val_preds_xgb = model_xgb.predict_proba(X_val)

auc_xgb = roc_auc_score(y_val, val_preds_xgb)
print(f'Validation ROC AUC (XGBoost): {auc_xgb}')

Validation ROC AUC (XGBoost): 0.9666887908278409


In [9]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train['comment_text'])
y_train = train.drop(columns=['comment_text'])
X_test = vectorizer.transform(test['comment_text'])

model_rf = OneVsRestClassifier(RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
))
model_lr = OneVsRestClassifier(LogisticRegression(
    random_state=42, 
    solver='liblinear',
    C=3,
    max_iter=2000,
    n_jobs=-1,
    class_weight='balanced'
))
model_xgb = OneVsRestClassifier(XGBClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.3,
    n_jobs=-1,
))

model_xgb.fit(X_train, y_train)
model_lr.fit(X_train, y_train)
model_rf.fit(X_train, y_train)

test_preds_xgb = model_xgb.predict_proba(X_test)
test_preds_lr = model_lr.predict_proba(X_test)
test_preds_rf = model_rf.predict_proba(X_test)

preds = (test_preds_xgb + test_preds_lr + test_preds_rf) / 3.0

print(preds)

d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1305: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 20.
  warnings.warn(
d:\VOAI25\

[[8.62870899e-01 3.67625282e-01 8.76358917e-01 2.54160032e-01
  8.21290100e-01 3.52167150e-01]
 [9.20491847e-03 6.89219185e-04 1.73710569e-03 5.78693051e-05
  3.59491577e-03 2.46146533e-03]
 [4.72209589e-02 5.50577555e-03 2.01612997e-02 1.10545398e-03
  2.40473120e-02 8.18553973e-03]
 ...
 [1.52196341e-02 1.12170551e-03 1.13064162e-02 4.73290151e-04
  4.09523653e-03 9.58590994e-04]
 [3.15081579e-02 7.92721291e-04 2.21594119e-02 1.04001537e-03
  9.57297286e-03 2.97014113e-02]
 [7.76763508e-01 8.66702946e-04 7.12664742e-01 7.54218804e-03
  4.59121488e-01 2.55893939e-02]]


In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'toxic': preds[:, 0],
    'severe_toxic': preds[:, 1],
    'obscene': preds[:, 2],
    'threat': preds[:, 3],
    'insult': preds[:, 4],
    'identity_hate': preds[:, 5]
})

submission.to_csv('submissions/tfidf_ens_voting.csv', index=False)